I trained this file on google collab with gpu(coz normally time consuming for to train with cpu to train all you have to do is upload processed_data.npz(213Mb),label_encoder.joblib to collab paste these code segments and run it took 52 mins to run this file)

Cell 1: Imports and Directory Setup

In [ ]:
from pathlib import Path
import sys
import time
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Set Paths
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT))

from src.models.ft_transformer import FTTransformer

PROCESSED_FILE = PROJECT / 'data' / 'processed_data.npz'
MODELS_DIR = PROJECT / 'models'
RESULTS_DIR = PROJECT / 'results' / 'ft_transformer'

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Cell 2: Random Seeds & Hardware Device

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', device)

Cell 3: Load Data & Class Names

In [ ]:
if not PROCESSED_FILE.exists():
    raise FileNotFoundError("Missing data/processed_data.npz. Run Notebook 01 first.")

processed_data = np.load(PROCESSED_FILE)
X_train = processed_data['X_train']
X_test = processed_data['X_test']
y_train = processed_data['y_train']
y_test = processed_data['y_test']

label_encoder = joblib.load(MODELS_DIR / 'label_encoder.joblib')
class_names = list(label_encoder.classes_)
number_of_classes = len(class_names)
input_size = X_train.shape[1]

print(f"Features: {input_size} | Classes: {number_of_classes} -> {class_names}")
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

Cell 4: Create DataLoaders

In [ ]:
BATCH_SIZE = 1024

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Cell 5: Initialize FT-Transformer Model

In [ ]:
model = FTTransformer(
    n_features=input_size,
    n_classes=number_of_classes,
    d_token=64,
    n_blocks=3,
    n_heads=4,
    ffn_factor=2.0,
    attention_dropout=0.15,
    ffn_dropout=0.15,
    residual_dropout=0.10,
    head_dropout=0.20,
    head_hidden_dim=64
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters: {total_params:,}")

Cell 6: Loss Function & Optimizer

In [ ]:
# Compute balanced class weights for imbalanced attack classes
class_counts = np.bincount(y_train, minlength=number_of_classes)
weight_values = len(y_train) / (number_of_classes * class_counts)
class_weights = torch.tensor(weight_values, dtype=torch.float32, device=device)

loss_function = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

Cell 7: Train the Model (15 Epochs)

In [ ]:
EPOCHS = 15
training_losses = []
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = loss_function(outputs, labels)
        loss.backward()

        # Gradient clipping for Transformer stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item() * len(labels)

    avg_loss = total_loss / len(train_dataset)
    training_losses.append(avg_loss)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} - Loss: {avg_loss:.4f}")

print(f"Training completed in {(time.time() - start_time) / 60:.2f} minutes.")

Cell 8: Save Model Weights

In [ ]:
torch.save(model.state_dict(), MODELS_DIR / 'ft_transformer_model.pt')
print(f"Saved checkpoint to: {MODELS_DIR / 'ft_transformer_model.pt'}")

Cell 9: Evaluate Test Predictions

In [ ]:
model.eval()
test_predictions = []
test_targets = []

with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(device)
        outputs = model(features)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_predictions.extend(preds)
        test_targets.extend(labels.numpy())

accuracy = accuracy_score(test_targets, test_predictions)
macro_f1 = f1_score(test_targets, test_predictions, average='macro')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Macro-F1: {macro_f1:.4f}\n")
print(classification_report(test_targets, test_predictions, target_names=class_names, zero_division=0))

Cell 10: Plot and Save Confusion Matrix

In [ ]:
cm = confusion_matrix(test_targets, test_predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('FT-Transformer Multiclass Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=300)
plt.show()
print(f"Plot saved to: {RESULTS_DIR / 'confusion_matrix.png'}")